In [29]:
!D:/AWS_CLI/aws.exe sso login --profile rmit

Attempting to open your default browser.
If the browser does not open, open the following URL:

https://oidc.ap-southeast-2.amazonaws.com/authorize?response_type=code&client_id=Q2DTXe-JRCw1BDUzRL6NBGFwLXNvdXRoZWFzdC0y&redirect_uri=http%3A%2F%2F127.0.0.1%3A62320%2Foauth%2Fcallback&state=42b28d74-3124-4e79-a11d-22d73cd5c469&code_challenge_method=S256&scopes=sso%3Aaccount%3Aaccess&code_challenge=KULsv2BUJVfajEpy00pNWXXOUgaNzHApV8uHOP19_E0
Successfully logged into Start URL: https://rmit-research.awsapps.com/start/#


In [35]:
import asyncio
import importlib.util
import os
from pathlib import Path

In [36]:
# Jupyter sets the current working directory to the folder containing the notebook.
# However, the original python script expects to run from the project root.
# We also make sure this behaves safely if the cell is run multiple times.
cwd = Path.cwd().resolve()
if cwd.name == "label" and cwd.parent.name == "scripts":
    os.chdir(cwd.parent.parent)
    print(f"Changed working directory to project root: {Path.cwd()}")
else:
    # Provide a fallback just in case it is already run from the project root
    print(f"Current working directory is already: {Path.cwd()}")

Current working directory is already: D:\Work\Research_Project\anaconda_research_project


In [37]:
# =========================================================
# Config
# =========================================================

# Existing script path (relative to the project root)
TARGET_SCRIPT = "trec_label_concurrent.py"

# Model settings
MAX_TOKENS = 2000
TARGET_PATH = Path("scripts") / "label" / TARGET_SCRIPT

# Languages only
LANGUAGES = [
    # "ru_instruct",
    # "zh_instruct",
    # "ga_instruct",
    # "ar_instruct",
    # "fr_instruct",
    # "vi_instruct",
    # "sw_instruct",
    # "ga_instruct",
    # "eng_instruct",
    # "hi_instruct",
    # "he_instruct",
    # "th_instruct",

    "rucwb",
    "zhcwb",
    "gacwb",
    "arcwb",
    "frcwb",
    "vicwb",
    "swcwb",
    "gacwb",
    "engcwb",
    "hicwb",
    "hecwb",
    "thcwb",

    # "ru_instruct_defended",
    # "fr_instruct_defended",
    # "eng_instruct_defended",
    # "vi_instruct_defended",
    # "zh_instruct_defended",
    # "ar_instruct_defended",
    # "he_instruct_defended",
    # "hi_instruct_defended",
    # "th_instruct_defended",
    # "ga_instruct_defended",
]

# Shared part range for all languages
START_PART = 1
END_PART = 6

In [38]:
# =========================================================
# Load target script dynamically
# =========================================================

if not TARGET_PATH.exists():
    raise FileNotFoundError(f"Could not find target script: {TARGET_PATH.resolve()}")

spec = importlib.util.spec_from_file_location("label_runner_target", TARGET_PATH)
module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(module)


def set_run_globals(mod, lang: str, start_part: int, end_part: int) -> None:
    mod.LANG = lang
    mod.START_PART = start_part
    mod.END_PART = end_part
    if hasattr(mod, "INFERENCE_CONFIG"):
        mod.INFERENCE_CONFIG["maxTokens"] = MAX_TOKENS
    else:
        mod.INFERENCE_CONFIG = {"maxTokens": MAX_TOKENS, "temperature": 0.0, "topP": 1.0}

    if lang == "raw":
        mod.PART_DIR = Path(f"retrieved/trec_dl_{mod.TREC_DL_YEAR}/judged/")
    else:
        mod.PART_DIR = Path(f"retrieved/trec_dl_{mod.TREC_DL_YEAR}/{lang}/")

    mod.PART_PATTERN = f"all_topics_trecdl_{mod.TREC_DL_YEAR}_part{{n}}.csv"

In [39]:
async def run_all_languages():
    for lang in LANGUAGES:
        print("\n" + "=" * 80)
        print(
            f"[RUNNER] Starting language={lang} | "
            f"parts={START_PART}..{END_PART}"
        )
        print("=" * 80)

        set_run_globals(module, lang, START_PART, END_PART)

        try:
            await module.main()
            print(f"[RUNNER] Finished language={lang}")
        except KeyboardInterrupt:
            print(f"\n[RUNNER] Interrupted while processing language={lang}")
            break
        except Exception as e:
            print(f"[RUNNER] Error while processing language={lang}: {e}")

# Run the main function. Notice we await directly since IPython already runs an event loop.
await run_all_languages()


[RUNNER] Starting language=rucwb | parts=1..6
[STOP] Press 'Q' to stop gracefully.

--- Running inference for model: meta.llama3-8b-instruct-v1:0 (run_id=20260326_234514, LANG=rucwb, mode=replace) ---
[STOP] Press 'Q' at any time to stop after the current in-flight items].
[all_topics_trecdl_2022_part2.csv] Loaded 536 rows
[HEADER] LANG='rucwb' | output columns = ['qid', 'query', 'pid', 'passage', 'relevance', 'unique_string', 'unique_string_ru', 'passage_injected', 'llm_relevance']
[CONCURRENCY] row_workers=2 queue_max=4
[all_topics_trecdl_2022_part3.csv] Loaded 806 rows
[HEADER] LANG='rucwb' | output columns = ['qid', 'query', 'pid', 'passage', 'relevance', 'unique_string', 'unique_string_ru', 'passage_injected', 'llm_relevance']
[CONCURRENCY] row_workers=2 queue_max=4
[all_topics_trecdl_2022_part5.csv] Loaded 534 rows in/out += 602/20 (totals 10482/340)all_topics_trecdl_2022_part3.csv] [14/806] tokens in/out += 620/20 (totals 8644/280)ll_topics_trecdl_2022_part2.csv] [8/536] tokens